# Hydraulic Predictive Maintenance — Stage 3: Model Development & Experiment Tracking

**Client:** Bosch Rexroth AG · **Stage:** 3 of 7 · **Notebook:** `03_Model_Development_and_Experiment_Tracking.ipynb`
**Builds on:** Stage 2 dataset, feature list and folds (`v0.2.0-stage2`) · **Code:** `src/pdm/models`, `src/pdm/train.py`

---

## Purpose

Stage 2 delivered a model-ready table: about 10,700 eligible machine-hours, 171 screened features, rebuilt targets and fixed leave-one-machine-out (LOMO) folds. Stage 3 answers three questions on machines the model has **never seen**:

1. **Which failure mode is developing?** (classification)
2. **How many hours until failure?** (RUL regression, capped at 14 days)
3. **Will this machine fail in the next 7 days, and how early can we say so without flooding the site with false alarms?** (early warning)

Every run is tracked in **MLflow** with its parameters, metrics, feature list, git commit, config hash and dataset fingerprint. The selected model per task is explained with **SHAP** and registered in the **MLflow Model Registry**, ready for Stage 4.

## How the Stage 2 plan is implemented

| Stage 2 plan item | Implementation | Module |
|---|---|---|
| 1. MLflow tracking | one parent run per training call, one nested run per task × model; tags: git commit, config hash, dataset hash | `models/tracking.py` |
| 2. Baselines first | EDA §8.3 rule detector (alarm + mode from the rule that fired), logistic regression; constant and ridge for RUL | `models/baselines.py`, `models/estimators.py` |
| 3. Failure-mode classification | LightGBM and XGBoost, balanced class weights; macro-F1 and confusion matrix per LOMO fold | `models/cv.py` |
| 4. RUL regression | LightGBM and XGBoost on the capped RUL (censored rows excluded); MAE, RMSE, asymmetric NASA score | `models/cv.py`, `models/metrics.py` |
| 5. Early warning | calibrated P(fail within 7 d); calibration and alarm threshold chosen by an **inner** LOMO; lead time at a fixed false-alarm rate | `models/cv.py` |
| 6. Sequence model (optional) | not built in this stage; see §9 | — |
| 7. Explainability and registry | SHAP (global and per alert); pyfunc models registered, `@champion` alias only if every baseline is beaten | `models/explain.py`, `models/registry.py` |

## Contents
1. Environment and project setup
2. Evaluation protocol
3. Stage 2 inputs
4. Training run (all tasks, tracked in MLflow)
5. Experiment tracking
6. Failure-mode classification
7. RUL regression
8. Early warning
9. Sequence model: decision
10. Explainability (SHAP)
11. Model Registry
12. Summary and plan for Stage 4

## 1. Environment and project setup

**Google Colab:** the project folder `bosch_rexroth_pdm/` on Google Drive, with the three CSVs in `data/raw/`, as in notebook 02. The cell mounts Drive, installs the dependencies (now including LightGBM, XGBoost, MLflow and SHAP) and the `pdm` package.
Locally, the cell just locates the project root. The MLflow store is `mlflow.db` + `mlartifacts/` in the project root (both git-ignored, like the data).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/bosch_rexroth_pdm")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)], check=True)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
print("Project root:", PROJECT_DIR)

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

import mlflow
import pdm
from pdm.config import load_config
from pdm.models.data import load_model_data, classification_rows, rul_rows, warning_rows
from pdm.models.tracking import tracking_uri
from pdm.train import run as train_run, PRIMARY, CANDIDATES

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("pdm", pdm.__version__, "| mlflow", mlflow.__version__, "| pandas", pd.__version__)
FIG_DIR = Path("reports/figures")
show = lambda name, width=900: display(Image(filename=str(FIG_DIR / f"stage3_{name}.png"), width=width))

### 1.1 Quality gate: run the unit tests first

The suite now has 41 tests. The 14 new Stage 3 tests check, among others, that **no model is ever trained on the machine it scores** (a spy on every early-warning fit), that the alarm threshold is the lowest one meeting the false-alarm target, that the NASA score punishes late predictions more than early ones, and that a model registered in MLflow predicts exactly what it predicted before logging.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "--color=no", "-p", "no:cacheprovider", "-W", "ignore"], capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1])
assert r.returncode == 0, "Unit tests failed: fix before training"


## 2. Evaluation protocol

**Leave-one-machine-out, out-of-fold.** Fold *k* holds out one machine completely; the model that scores a machine never saw any hour of it. All metrics below are pooled over the ten held-out machines unless stated otherwise.

**Rows per task** (all restricted to Stage 2 `eligible` rows: no repair downtime, no baseline period, no standby):

| Task | Target | Rows used | Primary metric (selection and gate) |
|---|---|---|---|
| Failure mode | `failure_mode_label` (healthy + 4 modes) | eligible | **macro-F1 on the folds whose mode was seen in training** |
| RUL | `rul_hours` (cap 336 h) | eligible, RUL known (censored rows excluded) | **MAE in the last 14 days before a failure** (true RUL < cap) |
| Early warning | `fail_within_h` (≤ 168 h) | eligible, target known | **mean lead time per failure event** (a missed event counts 0 h) at a fixed false-alarm rate |

**Why these headline metrics.**
* *Macro-F1* weights the rare modes (contamination 240 h, cylinder drift 192 h) as much as healthy hours. The HPU_06 fold (the only cylinder-drift machine) cannot be classified by name, so its detection rate is reported **separately**, not averaged away.
* *RUL on the critical window:* about two thirds of known-RUL rows sit at the 336 h cap. An error averaged over all rows mostly measures how well the model says "not soon"; the last 14 days are where maintenance is planned.
* *NASA score (PHM08):* with d = (predicted − true) / 24 h, a late prediction costs exp(d/10) − 1 and an early one exp(−d/13) − 1. Promising more life than the machine has is the expensive error.
* *Lead time at a fixed false-alarm rate:* a longer warning is worthless if bought with more false alarms. The operating point is `target_false_alarm_rate = 0.2 %` of healthy hours in alarm (≈ one alarm-hour per three weeks per machine), with 3 consecutive hours needed to raise an alarm. If the EDA rules achieve a **lower** false-alarm rate, the models are re-evaluated at the rules' rate so the comparison is always like for like.

**Early warning without leakage.** Calibration (isotonic) and the alarm threshold are fitted on an **inner** LOMO over the nine training machines of each outer fold; the held-out machine influences neither. The deployed model uses the calibrator and threshold fitted on the outer out-of-fold scores of all ten machines.

**No hyperparameter tuning.** With nine failure events, tuning on the LOMO score would fit the evaluation itself. The gradient-boosting settings are fixed, conservative defaults (`configs/config.yaml → modelling.models`), and the comparison is between model *families*, not tuned variants.

**Gate.** A model gets the registry alias `@champion` only if it beats **every** baseline of its task on the primary metric (and, for early warning, stays within the false-alarm budget).

## 3. Stage 2 inputs

Stage 3 reads exactly what Stage 2 persisted. If the processed files are missing (for example on a fresh Colab runtime), the Stage 2 pipeline is run first.

In [ ]:
cfg = load_config("configs/config.yaml")
if not (cfg["paths"]["processed_dir"] / "dataset.parquet").exists():
    from pdm.pipeline import run as pipeline_run
    pipeline_run(cfg, verbose=False)
data = load_model_data(cfg)
ds = data.ds
print(f"Dataset {data.dataset_hash}: {len(ds):,} machine-hours, {len(data.features)} features")
counts = pd.DataFrame({"classification": classification_rows(ds).sum(), "RUL (known)": rul_rows(ds).sum(),
                       "early warning": warning_rows(ds).sum()}, index=["rows"])
display(counts)
display(ds[classification_rows(ds)].groupby("failure_mode_label").agg(hours=("timestamp", "size"), machines=("machine_id", "nunique")).T)
display(data.events)

## 4. Training run (all tasks, tracked in MLflow)

`pdm.train.run` is the same function as the command line `python -m pdm.train`. It evaluates every baseline and model for each task, logs each as a nested MLflow run, selects and gates the best model, fits it on all ten machines, explains it and registers it. The notebook then reads the results back, so the notebook and the CLI cannot diverge.

Runtime is about 8 minutes on a 2-core machine (the early-warning task fits 100 models per candidate because of the nested protocol); allow longer on a free Colab runtime.

In [ ]:
report = train_run(cfg, data_label="real")
print("MLflow parent run:", report["mlflow_parent_run_id"], "| runtime", report["runtime_s"], "s")
pd.DataFrame({t: e["gate"] for t, e in report["tasks"].items()}).T

## 5. Experiment tracking

Every evaluation above is an MLflow run. The table is queried from the tracking store, not from the in-memory report: this is what a reviewer sees in `mlflow ui --backend-store-uri sqlite:///mlflow.db` (or `make mlflow-ui`). Per-fold metrics are stored as step series (`fold_<metric>`, step = fold), and each run carries the git commit, config hash and dataset fingerprint, so any number can be traced to the exact code, settings and data that produced it.

In [ ]:
mlflow.set_tracking_uri(tracking_uri(cfg))
runs = mlflow.search_runs(experiment_names=[cfg["modelling"]["tracking"]["experiment"]],
                          filter_string=f"tags.mlflow.parentRunId = '{report['mlflow_parent_run_id']}'")
keep = ["tags.mlflow.runName", "tags.git_commit", "tags.config_hash", "tags.dataset_hash",
        "metrics.macro_f1_seen", "metrics.mae_critical", "metrics.lead_time_mean_h", "metrics.false_alarm_rate"]
view = runs[[c for c in keep if c in runs.columns]].rename(columns=lambda c: c.split(".", 1)[1])
view["git_commit"] = view["git_commit"].str[:8]
view.sort_values("mlflow.runName" if "mlflow.runName" in view else view.columns[0]).round(4)

## 6. Failure-mode classification

Pooled out-of-fold results. *macro-F1 (seen folds)* excludes the HPU_06 fold, whose mode is absent from training; *HPU_06 detection* is the share of HPU_06's degradation hours flagged as **any** failure mode there. *Events correct* reads the model the way an engineer would: over each degradation window, which mode did the model point to most often?

In [ ]:
def table(task, cols):
    rows = {n: {c: m["summary"].get(c) for c in cols} for n, m in report["tasks"][task]["models"].items()}
    return pd.DataFrame(rows).T

t = table("failure_mode", ["macro_f1_seen", "macro_f1_all", "accuracy_seen", "detection_recall",
                           "healthy_false_positive_rate", "unseen_fold5_detection_recall", "events_mode_correct"])
t.round(3)

In [ ]:
best = report["tasks"]["failure_mode"]["gate"]["selected"]
show(f"failure_mode_{best}_confusion", 560)
show("failure_mode_per_fold")

In [ ]:
ev = pd.DataFrame(report["tasks"]["failure_mode"]["events"][best])
ev.round(2)

**Reading the results.** The confusion matrix is row-normalised (each true class sums to 100 %). The two errors that matter most are *degradation predicted as healthy* (a missed failure, the left column) and *healthy predicted as a failure mode* (a false alarm, the top row). Mixing up two failure modes is less costly: the machine still gets inspected, and SHAP (§10) shows the engineer which signals drove the call.

The HPU_06 row can only ever be *detected* in its own fold, never named, because no other machine has cylinder drift. That limitation is structural (one event), not a modelling choice, and more cylinder-drift events from the CMMS are the only fix.

## 7. RUL regression

Censored rows (no failure observed within 14 days of the end of data) are excluded from training and scoring. Predictions are clipped to [0, 336 h].

In [ ]:
table("rul", ["mae_critical", "rmse_critical", "nasa_critical", "mae_all", "rmse_all", "nasa_all", "n_critical"]).round(3)

In [ ]:
show("rul_traces", 1000)
show("rul_per_fold")

**Reading the results.** The constant model shows how much a model must learn to be useful at all. The traces show the typical shape of a capped-RUL model: flat at the cap while the machine is healthy, then a descent once degradation becomes visible. The descent starts only when the degradation *signal* starts, which for fast modes (contamination, 5-day windows) is late, so the error in the last 14 days is dominated by the first days of each window. A lower NASA score than MAE ranking would suggest means the model errs on the early (safe) side.

## 8. Early warning

Operating-point metrics (thresholds chosen per fold on the training machines). *Precision* and *recall* are hourly against `fail_within_h`; *lead time* is per failure event, from the first alarm inside the event's warning window (the earlier of the degradation start and failure − 7 d) to the failure. Alarms before that window count as false alarms.

In [ ]:
table("early_warning", ["lead_time_mean_h", "lead_time_median_h", "events_detected", "events_total", "false_alarm_rate",
                        "false_alarm_episodes_per_1000h", "precision", "recall", "pr_auc", "roc_auc", "brier"]).round(4)

In [ ]:
show("early_warning_lead_times", 1000)
show("early_warning_far_curve", 1000)

In [ ]:
ew_best = report["tasks"]["early_warning"]["gate"]["selected"]
pd.DataFrame(report["tasks"]["early_warning"]["per_fold"][ew_best]).round(4)

**Reading the results.** The per-fold table shows the threshold each fold chose on its own training machines and the false-alarm rate it achieved there (`inner_far`) versus on the held-out machine (`far`). A held-out rate well above the inner one would mean the threshold does not transfer between machines, which is the main deployment risk for any fleet-wide alarm.

The trade-off curve is **descriptive**: it sets one threshold on the pooled out-of-fold scores, so it is slightly optimistic compared with the per-fold operating point in the table above. The square marks the EDA rule detector, which has a single operating point.

Two structural limits apply to every model here. Contamination has a 5-day degradation window inside a 7-day target, so the first 2 days of each positive window carry no signal; and cylinder drift is again unseen in its own fold.

## 9. Sequence model: decision

The Stage 2 plan listed a 1D-CNN / LSTM on the hourly sequence as optional, to be kept only if it clearly beats gradient boosting. It was **not built** in this stage:

* **Too few sequences to learn from.** A sequence model learns the shape of degradation trajectories. There are 9 trajectories in total, and 8 in any training fold. Boosting on engineered features needs far fewer examples because the temporal shape is already encoded in the 6/24/72 h rolling means, slopes, lags and spectral features.
* **The evaluation could not show a *clear* win.** With 9 events, the LOMO lead-time and F1 estimates move by a whole event at a time; a difference between two strong models would be inside that noise.
* **Cost.** A deep-learning stack (PyTorch) would add a heavy dependency to the Stage 5 container for no demonstrated gain.

Revisit when the 2024 CMMS export adds failure events (the same request as for cylinder drift).

## 10. Explainability (SHAP)

SHAP values are computed on the final models (trained on all machines). For the classifier they are shown per failure mode; units are log-odds. The calibrator of the early-warning model is monotone, so features that raise the log-odds also raise the calibrated probability.

In [ ]:
show("shap_failure_mode", 1100)
pd.read_csv("reports/stage3_shap_failure_mode.csv", index_col=0).head(15).round(4)

**Why did an alert fire?** The example below takes the event with the longest warning and explains the model's score at the hour of the first alarm. This is the view the Stage 6 dashboard will show next to each alert: the top contributing features, their current values, and the direction in which each pushed the score.

In [ ]:
if (FIG_DIR / "stage3_shap_alert_example.png").exists():
    show("shap_alert_example", 800)
    display(pd.read_csv("reports/stage3_shap_alert_example.csv")[["rank", "feature", "value", "shap", "description"]].round(4))

## 11. Model Registry

Each task's selected model is logged as an `mlflow.pyfunc` model that bundles the estimator, the exact feature list and order, the class labels, and for early warning the calibrator, threshold and persistence rule. It is registered under `hpu-failure-mode`, `hpu-rul` and `hpu-early-warning`. The alias `@champion` is set only when the model passed the gate on real data; Stage 4 CI and the Stage 5 service load `models:/<name>@champion`.

In [ ]:
client = mlflow.MlflowClient()
reg = []
for task, name in cfg["modelling"]["registry"]["names"].items():
    for v in client.search_model_versions(f"name='{name}'"):
        mv = client.get_model_version(name, v.version)          # search results do not carry aliases
        reg.append({"name": name, "version": int(mv.version), "aliases": ", ".join(mv.aliases), **mv.tags})
pd.DataFrame(reg).sort_values(["name", "version"])

In [ ]:
# Load the registered model exactly as the Stage 5 service will, and score the latest hours of one machine
name = cfg["modelling"]["registry"]["names"]["early_warning"]
entry = report["tasks"]["early_warning"]["registered"]
uri = f"models:/{name}@{entry['alias']}" if entry.get("alias") else f"models:/{name}/{entry['version']}"
model = mlflow.pyfunc.load_model(uri)
latest = ds[ds.machine_id == data.events.iloc[0].machine_id].tail(6)
model.predict(latest[data.features]).assign(timestamp=latest.timestamp.values).round(3)

## 12. Summary and plan for Stage 4

The table below is generated from the run above (`reports/stage3_model_comparison.md`, also logged to MLflow).

In [ ]:
display(Markdown(Path("reports/stage3_model_comparison.md").read_text()))

### What Stage 3 delivers
- A **tested modelling package** (`src/pdm/models`, 14 new tests, 41 in total) and a CLI, `python -m pdm.train`.
- **Baselines and models for all three tasks**, evaluated leave-one-machine-out with the unseen cylinder-drift fold reported separately.
- A **leakage-free early-warning protocol**: calibration and alarm threshold chosen on the training machines of each fold, lead time reported at a fixed false-alarm rate.
- **MLflow tracking** of every run (git commit, config hash, dataset fingerprint, per-fold metrics, predictions, figures) and a **Model Registry** with a gated `@champion` alias.
- **SHAP explanations**, globally per failure mode and per alert.
- A **synthetic data generator** (`python -m pdm.synthetic`) with the real schema and EDA characteristics, so CI can run the whole pipeline without the private data. Its metrics are never reported.

### Open points for the client
| Topic | Why it matters | Request |
|---|---|---|
| More failure events (CMMS 2024) | 9 events limit every estimate; cylinder drift has 1 | export of all 2023–2024 failures with timestamps and modes |
| False-alarm budget | the operating point (0.2 % of healthy hours) is our assumption | the site's tolerated false alarms per machine per month |
| Cost of a late vs early prediction | sets the NASA-score constants and the alarm threshold | production-loss rate per downtime hour |

### Proposed Stage 4: CI/CD with GitHub Actions (for your confirmation)
1. **CI on every pull request:** lint, the 41 unit tests, and an end-to-end smoke run (`pdm.synthetic` → `pdm.pipeline` → `pdm.train --data-label synthetic --no-register`) on the synthetic data, in a few minutes.
2. **Data-validation job:** the Stage 2 validation gate on each new data batch, publishing the validation report as a build artifact.
3. **Training workflow (manual / scheduled):** run on real data with access to the data store and the MLflow server, and fail if the new model does not pass the gate against the current `@champion`.
4. **Model promotion:** move the `@champion` alias only through a reviewed workflow, recording the git commit and dataset fingerprint.
5. **Remote MLflow tracking server** (replacing the local SQLite store) so runs from CI, Colab and laptops land in one place.